# Project 64 — Local Tool Selection Benchmark
## Evaluate LLM Tool-Routing Accuracy with Ground Truth

**Stack:** LangChain · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## Step 1 — Define Tool Registry & Test Suite

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import pandas as pd, json, time

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

tool_registry = [
    {"name": "calculator",      "desc": "Perform math calculations and unit conversions"},
    {"name": "web_search",      "desc": "Search the internet for current information"},
    {"name": "file_reader",     "desc": "Read and display file contents"},
    {"name": "code_runner",     "desc": "Execute Python code snippets"},
    {"name": "database_query",  "desc": "Run SQL queries against a database"},
    {"name": "email_sender",    "desc": "Compose and send emails"},
    {"name": "calendar",        "desc": "Schedule meetings and check availability"},
    {"name": "translator",      "desc": "Translate text between languages"},
]

test_cases = [
    ("What is 15% of $230?", "calculator"),
    ("Find recent papers on transformers", "web_search"),
    ("Show me the README.md file", "file_reader"),
    ("Execute the data processing script", "code_runner"),
    ("How many orders were placed last month?", "database_query"),
    ("Send the weekly report to the team", "email_sender"),
    ("Am I free at 3pm on Thursday?", "calendar"),
    ("Translate 'hello world' to Japanese", "translator"),
    ("Calculate compound interest on $5000 at 4% for 3 years", "calculator"),
    ("What files are in the /data directory?", "file_reader"),
    ("Schedule a meeting with Alice for next Tuesday", "calendar"),
    ("What's the current stock price of AAPL?", "web_search"),
    ("Run the test suite and show results", "code_runner"),
    ("How many users signed up this week?", "database_query"),
    ("Notify the team about the deployment", "email_sender"),
    ("Convert this paragraph to Spanish", "translator"),
]
print(f"Registry: {len(tool_registry)} tools | Test suite: {len(test_cases)} cases")

## Step 2 — Run Benchmark

In [ ]:
class ToolChoice(BaseModel):
    selected_tool: str = Field(description="Tool name from registry")
    confidence: float = Field(ge=0, le=1)
    reasoning: str

selector = llm.with_structured_output(ToolChoice)
tool_desc_block = "\n".join([f"- {t['name']}: {t['desc']}" for t in tool_registry])

results = []
for query, expected in test_cases:
    start = time.time()
    choice = selector.invoke(
        f"Available tools:\n{tool_desc_block}\n\nUser request: {query}\nSelect the best tool."
    )
    elapsed = time.time() - start
    correct = choice.selected_tool.strip().lower() == expected.lower()
    results.append({
        "query": query[:45], "expected": expected,
        "selected": choice.selected_tool, "correct": correct,
        "confidence": choice.confidence, "latency": round(elapsed, 2),
    })

df = pd.DataFrame(results)
accuracy = df["correct"].mean()
print(f"Overall accuracy: {accuracy:.0%} ({df['correct'].sum()}/{len(df)})")

## Step 3 — Error Analysis

In [ ]:
print("DETAILED RESULTS")
print("=" * 70)
for _, row in df.iterrows():
    icon = "✓" if row["correct"] else "✗"
    print(f"  {icon} {row['query']:<47} exp={row['expected']:<15} got={row['selected']}")

# Misclassified
errors = df[~df["correct"]]
if len(errors) > 0:
    print(f"\nERROR ANALYSIS — {len(errors)} misclassifications:")
    for _, row in errors.iterrows():
        print(f"  '{row['query']}' → expected {row['expected']}, got {row['selected']}")
else:
    print("\nPerfect score — no misclassifications!")

# Confidence distribution
print(f"\nConfidence stats:")
print(f"  Mean:  {df['confidence'].mean():.2f}")
print(f"  Correct items avg confidence: {df[df['correct']]['confidence'].mean():.2f}")
if len(errors) > 0:
    print(f"  Wrong items avg confidence:   {errors['confidence'].mean():.2f}")

## Step 4 — Confusion Matrix & per-Tool Precision

In [ ]:
# Per-tool precision
tool_stats = []
for tool in [t["name"] for t in tool_registry]:
    predicted = df[df["selected"] == tool]
    expected_set = df[df["expected"] == tool]
    tp = len(predicted[predicted["correct"]])
    fp = len(predicted[~predicted["correct"]])
    fn = len(expected_set) - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    tool_stats.append({"tool": tool, "precision": precision, "recall": recall, "tp": tp, "fp": fp, "fn": fn})

stats_df = pd.DataFrame(tool_stats)
print("PER-TOOL PRECISION & RECALL")
print("=" * 60)
print(stats_df.to_string(index=False))

print(f"\nAvg latency: {df['latency'].mean():.2f}s")
print(f"P95 latency: {df['latency'].quantile(0.95):.2f}s")

## What You Learned
- **Tool selection benchmarking** with ground truth labels
- **Error analysis** for misclassified routing decisions
- **Per-tool precision & recall** metrics
- **Confidence calibration** analysis